# 02 · Data prep (finishes Phase 1: Steps 1.3–1.5)

**Runtime: CPU** (Runtime → Change runtime type → **CPU**, High RAM if offered). No GPU needed, so no GPU credits used.

One run that does every image-dependent part of Phase 1:
1. Builds the pinned Python environment (same versions as training).
2. Rebuilds the label files on Drive and checks they are **byte-identical** to the committed hashes.
3. Unzips all images to fast local disk.
4. Hashes every image (split manifests), measures sizes and EXIF rotation, and checks for duplicate files across splits.
5. Formats real examples with the model's chat template and measures token lengths at 4 image-size budgets.

Expect about **25–40 minutes**. Run cells top to bottom. **Paste back** notes say what to send.

Every command runs through `sh()`, which **stops with an error if the command fails**, so a failed step cannot silently break the later ones (R-007).

## 1 · Settings, Drive, code

In [ ]:
import os
import subprocess

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"  # zips (raw/) + label files (processed/)
LOCAL = "/content/data"  # unzipped images and annotations (wiped on disconnect)
LOGS = f"{DRIVE_ROOT}/logs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"


def sh(cmd, **env):
    """Run a shell command, stream its output, and raise if it fails."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}). Stop here and paste the output above."
        )


drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nproc && free -g | head -2")

**Paste back:** the commit line.

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Rebuild label files and verify against committed hashes (~1 min)
Writes `vqa_*.jsonl` / `quality_*.jsonl` to `MyDrive/vizwiz/datasets/processed/`. Uses the committed image fingerprints to keep copies of test photos out of training (D-038).

In [ ]:
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits")
sh(
    f"{PY} -m vizwiz.data.splits --annotations {LOCAL}/annotations --out {DATA}/processed --results /content/splits_rebuild --expect {REPO_DIR}/results/splits/files.json"
)

**Paste back:** the last two lines: `excluded 605 ...` and `REBUILD MATCHES COMMITTED HASHES`.

## 4 · Unzip all images to local disk (~4–5 min)

In [ ]:
%%time
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train val test")
sh("df -h /content | tail -1")

**Paste back:** the three `[extract]` lines.

## 5 · Hash and measure every image (~3–8 min)

In [ ]:
%%time
sh(
    f"{PY} -m vizwiz.data.images --images {LOCAL}/images --membership {REPO_DIR}/results/splits/membership.json --results {LOGS}/splits --expect {REPO_DIR}/results/splits/image_hashes.json"
)

**Paste back:** the first line (`membership: ... excluded: 605`), the `duplicates` block, and the last two lines (`IMAGE HASHES MATCH COMMITTED`, `IMAGE SCAN OK`).

## 6 · Format examples and measure token lengths (~10–20 min)
Downloads only the model's processor (a few MB), not the weights. First prints one **synthetic** example per task (no dataset text) showing the chat template and exactly which tokens are trained, then measures 200 real val examples per task at 4 image-size budgets.

In [ ]:
%%time
sh(
    f"{PY} -m vizwiz.data.formatting stats --records {DATA}/processed --images {LOCAL}/images --split val --sample 200 --report {LOGS}/token_stats.json",
    HF_HOME="/content/hf_cache",
)

**Paste back:** the three `=== demo ===` blocks with their `trained tokens` lines, and the last line (`ALL PREFIX CHECKS PASSED`).

## 7 · Flush Drive, then disconnect
Upload back from `MyDrive/vizwiz/logs/`:
- `splits/image_stats.json`
- `token_stats.json`

(`image_hashes.json` is not needed: cell 5 already checked it matches the committed one.)

Then Runtime → Disconnect and delete runtime.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")